# 🚦 Fine-tune YOLOv8 for Road-Hazard Detection

Fine-tunes YOLOv8 on a public pothole dataset using a **free Colab T4 GPU**, then exports `pothole.pt`. **No account or API key needed** — the dataset auto-downloads.

## How to run (3 clicks)
1. **Runtime → Change runtime type → Hardware accelerator: T4 GPU** → Save
2. **Runtime → Run all**
3. Wait ~5–10 min. At the end `pothole.pt` downloads to your computer.
4. Send that file back — it gets wired into the app as `roadsafety_ai/models/pothole.pt` (auto-loaded at startup).

In [ ]:
# 0) Confirm the free GPU is attached (should print a Tesla T4)
!nvidia-smi

In [ ]:
# 1) Install dependencies
!pip install -q ultralytics huggingface_hub
import ultralytics; ultralytics.checks()

In [ ]:
# 2) Auto-download the pothole dataset (public, no key) and fix its config
import os, yaml
from huggingface_hub import snapshot_download

ds = snapshot_download(
    repo_id="Ryukijano/Pothole-detection-Yolov8",
    repo_type="dataset",
    local_dir="pothole_dataset",
)

# Rewrite data.yaml with an absolute path + a real class name ('pothole')
cfg = {
    "path":  os.path.abspath("pothole_dataset"),
    "train": "train/images",
    "val":   "valid/images",
    "test":  "test/images",
    "nc":    1,
    "names": ["pothole"],
}
DATA_YAML = "pothole_dataset/data.yaml"
yaml.safe_dump(cfg, open(DATA_YAML, "w"))
print("\u2713 dataset ready:")
print(open(DATA_YAML).read())

In [ ]:
# 3) Fine-tune YOLOv8s (COCO-pretrained) on the pothole data
from ultralytics import YOLO

model = YOLO("yolov8s.pt")          # pretrained base
model.train(
    data=DATA_YAML,
    epochs=80,
    imgsz=640,
    batch=16,
    device=0,                       # T4 GPU
    name="pothole_model",
    patience=25,
    plots=True,
)

In [ ]:
# 4) Validate — check accuracy
metrics = model.val()
print("mAP50    :", round(float(metrics.box.map50), 4))
print("mAP50-95 :", round(float(metrics.box.map), 4))

In [ ]:
# 5) Export the fine-tuned weights and download them
import shutil
from ultralytics import YOLO
from google.colab import files

src = "runs/detect/pothole_model/weights/best.pt"
shutil.copy(src, "pothole.pt")
print("Trained classes:", YOLO(src).names)
print("\u2b07\ufe0f downloading pothole.pt \u2014 send this file back to wire it into the app")
files.download("pothole.pt")